In [3]:
# MQTT Subscribe 測試應用程式
# 這個 notebook 示範如何使用 paho-mqtt 套件來訂閱並接收 MQTT 訊息。

# 導入必要的套件
import paho.mqtt.client as mqtt
import time
import json

# MQTT Broker 設定（連接到本地 Raspberry Pi 的 MQTT Broker）
BROKER = "localhost"  # 或使用 "127.0.0.1"
PORT = 1883  # 標準 MQTT 埠，如果您的 Broker 使用不同埠號請修改
TOPIC = "客廳/溫度"  # 訂閱的主題名稱（與 lesson6_1.ipynb 相同）

# 建立 MQTT 客戶端（使用新的回調 API 版本 2）
client = mqtt.Client(callback_api_version=mqtt.CallbackAPIVersion.VERSION2)

# 連線回調函數（VERSION2 API）
def on_connect(client, userdata, flags, reason_code, properties):
    if reason_code.is_failure:
        print(f"❌ 連線失敗，錯誤代碼: {reason_code}")
    else:
        print(f"✅ 成功連接到 MQTT Broker: {BROKER}")
        # 連線成功後訂閱主題
        client.subscribe(TOPIC, qos=1)
        print(f"✅ 已訂閱主題: {TOPIC}")

# 訂閱回調函數（當成功訂閱時被調用）
def on_subscribe(client, userdata, mid, reason_codes, properties):
    print(f"✅ 訂閱確認 (mid: {mid})")
    print("📡 開始監聽訊息...\n")

# 接收訊息回調函數（VERSION2 API）
def on_message(client, userdata, message):
    topic = message.topic
    payload = message.payload.decode('utf-8')
    
    print(f"📨 收到訊息")
    print(f"   主題: {topic}")
    print(f"   內容: {payload}")
    
    # 嘗試解析 JSON 格式的訊息
    try:
        data = json.loads(payload)
        print(f"   📊 JSON 解析成功:")
        print(f"      - 裝置: {data.get('device', 'N/A')}")
        print(f"      - 溫度: {data.get('temperature', 'N/A')}°C")
        print(f"      - 濕度: {data.get('humidity', 'N/A')}%")
        if 'timestamp' in data:
            timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(data['timestamp']))
            print(f"      - 時間: {timestamp}")
    except json.JSONDecodeError:
        # 如果不是 JSON 格式，就當作普通文字訊息
        print(f"   📝 文字訊息")
    
    print("-" * 50)

# 設定回調函數
client.on_connect = on_connect
client.on_subscribe = on_subscribe
client.on_message = on_message

# 連接到 Broker
print(f"正在連接到 {BROKER}...")
client.connect(BROKER, PORT, 60)
client.loop_start()  # 開始背景執行緒處理訊息

# 等待連線建立
time.sleep(1)

# 持續監聽訊息（測試用：監聽 30 秒）
print("⏱️  將監聽訊息 30 秒...")
print("💡 提示：請在另一個終端執行 lesson6_1.ipynb 來發送測試訊息\n")

try:
    time.sleep(30)  # 監聽 30 秒
except KeyboardInterrupt:
    print("\n⚠️  使用者中斷")

print("\n✅ 監聽結束")

# 關閉連線
client.loop_stop()
client.disconnect()
print("✅ MQTT 連線已關閉")


正在連接到 localhost...
✅ 成功連接到 MQTT Broker: localhost
✅ 已訂閱主題: 客廳/溫度
✅ 訂閱確認 (mid: 1)
📡 開始監聽訊息...

⏱️  將監聽訊息 30 秒...
💡 提示：請在另一個終端執行 lesson6_1.ipynb 來發送測試訊息


⚠️  使用者中斷

✅ 監聽結束
✅ MQTT 連線已關閉


## 進階版：持續監聽並統計訊息


In [4]:
# 進階版：持續監聽並統計接收到的訊息

import paho.mqtt.client as mqtt
import time
import json
from datetime import datetime

# MQTT 設定
BROKER = "localhost"
PORT = 1883
TOPIC = "客廳/溫度"

# 統計變數
message_count = 0
json_count = 0
text_count = 0

# 建立客戶端
client = mqtt.Client(callback_api_version=mqtt.CallbackAPIVersion.VERSION2)

def on_connect(client, userdata, flags, reason_code, properties):
    if reason_code.is_failure:
        print(f"❌ 連線失敗，錯誤代碼: {reason_code}")
    else:
        print(f"✅ 成功連接到 MQTT Broker: {BROKER}")
        client.subscribe(TOPIC, qos=1)
        print(f"✅ 已訂閱主題: {TOPIC}\n")

def on_subscribe(client, userdata, mid, reason_codes, properties):
    print("📡 開始監聽訊息...")
    print("=" * 60)

def on_message(client, userdata, message):
    global message_count, json_count, text_count
    
    topic = message.topic
    payload = message.payload.decode('utf-8')
    message_count += 1
    
    current_time = datetime.now().strftime('%H:%M:%S')
    print(f"\n[{current_time}] 📨 訊息 #{message_count}")
    print(f"   主題: {topic}")
    print(f"   內容: {payload}")
    
    # 嘗試解析 JSON
    try:
        data = json.loads(payload)
        json_count += 1
        print(f"   📊 JSON 格式訊息")
        for key, value in data.items():
            if key == 'timestamp':
                value = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(value))
            print(f"      {key}: {value}")
    except json.JSONDecodeError:
        text_count += 1
        print(f"   📝 文字格式訊息")
    
    print(f"   📈 統計: 總計={message_count}, JSON={json_count}, 文字={text_count}")
    print("-" * 60)

# 設定回調
client.on_connect = on_connect
client.on_subscribe = on_subscribe
client.on_message = on_message

# 連接並開始監聽
print(f"正在連接到 {BROKER}...")
client.connect(BROKER, PORT, 60)
client.loop_start()
time.sleep(1)

# 監聽 60 秒
print("⏱️  將監聽訊息 60 秒...")
print("💡 提示：請在另一個終端執行 lesson6_1.ipynb 來發送測試訊息\n")

try:
    time.sleep(60)
except KeyboardInterrupt:
    print("\n⚠️  使用者中斷")

print("\n" + "=" * 60)
print("📊 最終統計結果:")
print(f"   總訊息數: {message_count}")
print(f"   JSON 訊息: {json_count}")
print(f"   文字訊息: {text_count}")
print("=" * 60)

# 關閉連線
client.loop_stop()
client.disconnect()
print("\n✅ MQTT 連線已關閉")


正在連接到 localhost...
✅ 成功連接到 MQTT Broker: localhost
✅ 已訂閱主題: 客廳/溫度

📡 開始監聽訊息...
⏱️  將監聽訊息 60 秒...
💡 提示：請在另一個終端執行 lesson6_1.ipynb 來發送測試訊息




📊 最終統計結果:
   總訊息數: 0
   JSON 訊息: 0
   文字訊息: 0

✅ MQTT 連線已關閉


## 使用說明

### 測試步驟

1. **先執行此 notebook（lesson6_2.ipynb）**：
   - 訂閱主題並開始監聽訊息
   - 程式會持續監聽 30 秒（基礎版）或 60 秒（進階版）

2. **再執行 lesson6_1.ipynb**：
   - 發布測試訊息到相同主題
   - 您應該會在此 notebook 中看到接收到的訊息

### 主題設定

- 確保兩個 notebook 使用相同的主題名稱（預設為 `"客廳/溫度"`）
- 可以修改 `TOPIC` 變數來測試不同的主題

### 訊息格式

- **文字訊息**：直接顯示內容
- **JSON 訊息**：自動解析並顯示結構化數據

### 停止監聽

- 等待設定的時間結束（自動停止）
- 或按 `Ctrl+C` 中斷執行（KeyboardInterrupt）

### 提示

💡 建議同時開啟兩個 notebook：
- 一個執行 lesson6_1.ipynb（發布者）
- 另一個執行 lesson6_2.ipynb（訂閱者）

這樣可以即時看到訊息發布和接收的效果！
